# Milestone 3

Nama  : Putri joeliya

Batch : CODA-020-RMT

ppt like : https://docs.google.com/presentation/d/1fh48pWIji35MN6b4BRvzoAmJ-RtO_pItkeKx110hMEI/edit?slide=id.g3f78683f4ee_0_581#slide=id.g3f78683f4ee_0_581


In this step, we download the IBM HR Analytics dataset from Kaggle using `kagglehub`. The raw dataset will be saved into the `data/` directory for further exploration and preprocessing.


## Data Extract From Kaggle

In [1]:
pip install kagglehub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import kagglehub
import os
import shutil
DATASET_ROOT_DIR = "data/"


path = kagglehub.dataset_download("pavansubhasht/ibm-hr-analytics-attrition-dataset")
print("Path to dataset files:", path)
if not os.path.exists(DATASET_ROOT_DIR):
    os.makedirs(DATASET_ROOT_DIR)
for item in os.listdir(path):
    shutil.copy(os.path.join(path, item), DATASET_ROOT_DIR)

print("succeed")

Path to dataset files: C:\Users\XPS\.cache\kagglehub\datasets\pavansubhasht\ibm-hr-analytics-attrition-dataset\versions\1
succeed


## Exploraroty Data Analysis (EDA)
### Data Overview & Quality Check
We inspect the dataset structure, missing values, duplicates, and column data types to understand the underlying distribution and quality of the raw data.

In [3]:
import pandas as pd
df = pd.read_csv('data/WA_Fn-UseC_-HR-Employee-Attrition.csv')

print('rows:', df.shape)

df.head()

rows: (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [5]:
print("Null:", df.isnull().sum().sum())

print("\nDuplicate:", df.duplicated().sum())

Null: 0

Duplicate: 0


### Data Cleaning & Transformation Strategy


In [6]:
# checking static/constant columns
constant_cols = [col for col in df.columns if df[col].nunique() == 1]
print("zero variance col: ", constant_cols)

zero variance col:  ['EmployeeCount', 'Over18', 'StandardHours']


In [7]:
# Remove static/constant columns
df_cleaned = df.drop(columns=['EmployeeCount', 'Over18', 'StandardHours'])
# Construct a composite unique identifier
df_cleaned['emplyee_ID_key'] = df_cleaned['EmployeeNumber'].astype(str) + '_' + df_cleaned['Age'].astype(str)
# Filter the dataset to retain records of working-age employees
df_cleaned = df_cleaned[(df_cleaned['Age'] >= 18) & (df_cleaned['Age'] <= 65)]
print("Ukuran data setelah di-clean:", df_cleaned.shape)
df_cleaned.head(3)

Ukuran data setelah di-clean: (1470, 33)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,...,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,emplyee_ID_key
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,...,1,0,8,0,1,6,4,0,5,1_41
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,...,4,1,10,3,3,10,7,1,7,2_49
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,...,2,0,7,3,3,0,0,0,0,4_37


### Load Clean Data 

In [8]:
df_cleaned.to_csv("data/transformed_data_clean.csv", index=False)

GX

## Data Validation with Great Expectations (GX)

In [9]:
%pip install -q "great-expectations==0.18.19"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from great_expectations.data_context import FileDataContext

context = FileDataContext.create(project_root_dir='./')

### Setup Data Context, Datasource, and Data Asset
We initialize the Great Expectations context and define our Pandas Datasource along with a CSV Data Asset pointing to our cleaned dataset (`transformed_data_clean.csv`).

In [11]:
datasource_name = "csv-data-hr"
datasource = context.sources.add_or_update_pandas(datasource_name)

asset_name = "hr-cleaned-data"
path_to_data = "data/transformed_data_clean.csv"
asset = datasource.add_csv_asset(asset_name, filepath_or_buffer=path_to_data)

batch_request = asset.build_batch_request()

### Create Expectation Suite and Validator
We create a new Expectation Suite and attach a Validator to read the batch request generated from our Data Asset.

In [12]:
expectation_suite_name = 'expectation-hr-dataset'
context.add_or_update_expectation_suite(expectation_suite_name)

validator = context.get_validator(
    batch_request = batch_request,
    expectation_suite_name = expectation_suite_name
)

validator.head()

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,...,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,emplyee_ID_key
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,...,1,0,8,0,1,6,4,0,5,1_41
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,...,4,1,10,3,3,10,7,1,7,2_49
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,...,2,0,7,3,3,0,0,0,0,4_37
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,...,3,0,8,3,3,8,7,3,0,5_33
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,...,4,1,6,3,3,2,2,2,2,7_27


### Defining Expectations

In [13]:
# 1. Unique ID
validator.expect_column_values_to_be_unique(column="emplyee_ID_key")



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 1470,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [14]:
# 2. Age Range Valid
validator.expect_column_values_to_be_between(column="Age", min_value=18, max_value=65)



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 1470,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [15]:
# 3. Gender Value Set
validator.expect_column_values_to_be_in_set(column="Gender", value_set=["Male", "Female"])



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 1470,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [16]:
validator.expect_column_values_to_be_of_type(column="MonthlyIncome", type_="int64")

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "observed_value": "int64"
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [17]:
# 5. Attrition Non-Null Check
validator.expect_column_values_to_not_be_null(column="Attrition")



Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 1470,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [18]:
# 6. Column Existence Check (Department)
validator.expect_column_values_to_be_in_set(
    column="Department", 
    value_set=["Sales", "Research & Development", "Human Resources"]
)



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 1470,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [19]:
# 7. OverTime String Length Range ('Yes' / 'No')
validator.expect_column_value_lengths_to_be_between(column="OverTime", min_value=2, max_value=3)



Calculating Metrics:   0%|          | 0/9 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 1470,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [20]:
# 8. BusinessTravel Value Set
validator.expect_column_values_to_be_in_set(
    column="BusinessTravel", 
    value_set=["Travel_Rarely", "Travel_Frequently", "Non-Travel"]
)



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 1470,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [21]:
# 9. Education Level Range (Skala 1 - 5)
validator.expect_column_values_to_be_between(column="Education", min_value=1, max_value=5)



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 1470,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [22]:
# 10. Environment Satisfaction Rating Range (Skala 1 - 4)
validator.expect_column_values_to_be_between(column="EnvironmentSatisfaction", min_value=1, max_value=4)



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 1470,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [23]:
# 11. Marital Status Value Set
validator.expect_column_values_to_be_in_set(
    column="MaritalStatus", 
    value_set=["Single", "Married", "Divorced"]
)



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 1470,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [24]:
# 12. Job Level Range (Skala 1 - 5)
validator.expect_column_values_to_be_between(column="JobLevel", min_value=1, max_value=5)



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 1470,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [25]:
# 13. Work Life Balance Rating Range (Skala 1 - 4)
validator.expect_column_values_to_be_between(column="WorkLifeBalance", min_value=1, max_value=4)



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 1470,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [26]:
# 14. Distance From Home Non-Negative Check
validator.expect_column_values_to_be_between(column="DistanceFromHome", min_value=1, max_value=100)



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 1470,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [27]:
# 15. Total Working Years Minimum Boundary Check
validator.expect_column_values_to_be_between(column="TotalWorkingYears", min_value=0, max_value=50)




Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 1470,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [28]:

validator.save_expectation_suite(discard_failed_expectations=False)



### Running Checkpoint & Building Data Docs
Finally, we execute the validation Checkpoint to verify all expectations and generate interactive HTML Data Docs to review the quality report.

In [29]:
checkpoint = context.add_or_update_checkpoint(
    name="m3_checkpoint",
    validator=validator
)



In [30]:
checkpoint_result = checkpoint.run()

Calculating Metrics:   0%|          | 0/100 [00:00<?, ?it/s]

In [31]:
context.build_data_docs()
# context.open_data_docs()

{'local_site': 'file://c:\\Users\\XPS\\OneDrive\\Documents\\CODA\\CODA 0.1\\m3\\gx\\uncommitted/data_docs/local_site/index.html'}